# 第2课 图像预处理

适合对象：七年级同学（已经会运行基础 Python 代码）

前置知识：
- 会打开 Jupyter Notebook
- 知道图片由很多像素组成
- 了解 `for` 循环和变量

学习目标：
- 理解什么是图像标准化（让像素数值更适合 AI 学习）
- 学会把彩色图转成灰度图
- 学会把灰度图做二值化（黑白图）
- 学会做基础数据增强（旋转、翻转、亮度变化）

本课提示：如果本地没有测试图片，Notebook 会自动生成一张练习图。


## 课程流程

1. 准备图片并显示
2. 图像标准化（Normalization）
3. 灰度转换（RGB -> Gray）
4. 二值化（Gray -> Black/White）
5. 数据增强（Data Augmentation）
6. 课堂练习


In [ ]:
# 第1步：导入需要的库
# Pillow: 读取和处理图片
# numpy: 做数值计算
# matplotlib: 把图片画出来
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance, ImageDraw

# 为了让每次运行结果更稳定，我们固定随机种子
random.seed(7)
np.random.seed(7)

def load_demo_image() -> tuple[Image.Image, str]:
    # 优先读取本地图片；如果没有，就自动生成一张示例图。
    candidates = [
        Path('images/sample_building.jpg'),
        Path('../test.jpg'),
        Path('../result_sketch.jpg'),
    ]
    for p in candidates:
        if p.exists():
            return Image.open(p).convert('RGB'), f'已加载本地图片: {p}'

    # 没有找到图片时，生成一张“天空+建筑”的练习图
    w, h = 320, 240
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    arr[..., 0] = np.linspace(80, 20, h, dtype=np.uint8)[:, None]   # 红色通道
    arr[..., 1] = np.linspace(170, 80, h, dtype=np.uint8)[:, None]  # 绿色通道
    arr[..., 2] = np.linspace(255, 130, h, dtype=np.uint8)[:, None] # 蓝色通道

    img = Image.fromarray(arr)
    draw = ImageDraw.Draw(img)
    draw.rectangle((70, 70, 250, 220), fill=(60, 60, 70), outline=(255, 255, 255), width=2)
    for y in range(90, 210, 24):
        for x in range(90, 240, 32):
            draw.rectangle((x, y, x + 14, y + 10), fill=(240, 230, 170))
    draw.text((12, 12), 'Demo Building', fill=(255, 255, 255))
    return img, '未找到本地图片，已自动生成示例图'

img_rgb, image_note = load_demo_image()
print(image_note)
print('图片尺寸:', img_rgb.size, '| 颜色模式:', img_rgb.mode)

plt.figure(figsize=(5, 4))
plt.imshow(img_rgb)
plt.title('原始图片')
plt.axis('off')
plt.show()


## Step 1 图像标准化

为什么要标准化？
- 原始像素通常是 `0~255`，范围比较大。
- 神经网络更喜欢“范围比较小、分布比较稳定”的输入。
- 常见做法：先除以 255 变成 `0~1`，再做“减均值、除标准差”。


In [ ]:
# 第2步：把 PIL 图片转成 numpy 数组，并做标准化
img_np = np.array(img_rgb).astype(np.float32)

# 2.1 把 0~255 缩放到 0~1
img_01 = img_np / 255.0

# 2.2 计算每个通道的均值和标准差（R/G/B 分开算）
mean = img_01.mean(axis=(0, 1), keepdims=True)
std = img_01.std(axis=(0, 1), keepdims=True) + 1e-6  # 防止除以0

# 2.3 标准化：得到“均值约为0、方差约为1”的数据
img_norm = (img_01 - mean) / std

print('缩放后范围: [%.3f, %.3f]' % (img_01.min(), img_01.max()))
print('标准化后范围: [%.3f, %.3f]' % (img_norm.min(), img_norm.max()))
print('标准化后均值: %.3f' % img_norm.mean())

# 为了可视化，把标准化结果临时映射回可显示范围
img_norm_vis = np.clip((img_norm * 0.25 + 0.5), 0, 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_01)
plt.title('缩放到 0~1')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_norm_vis)
plt.title('标准化后(仅用于显示)')
plt.axis('off')

plt.tight_layout()
plt.show()


## Step 2 灰度转换

灰度图只有“明暗”信息，不包含颜色。

为什么有用？
- 对很多任务（例如边缘、轮廓）来说，颜色不是最关键的信息。
- 灰度图数据量更小，处理更快。


In [ ]:
# 第3步：把 RGB 彩图转换成灰度图
img_gray = ImageOps.grayscale(img_rgb)
gray_np = np.array(img_gray)

print('灰度图尺寸:', img_gray.size)
print('灰度图数组形状:', gray_np.shape)
print('灰度像素范围:', gray_np.min(), '~', gray_np.max())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title('RGB 彩图')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')
plt.title('灰度图')
plt.axis('off')

plt.tight_layout()
plt.show()


## Step 3 二值化

二值化的思想：
- 设定一个阈值 `T`
- 灰度值 `>= T` 的像素变白（255）
- 灰度值 `< T` 的像素变黑（0）

这一步常用于提取轮廓和分割目标。


In [ ]:
# 第4步：试试不同阈值的二值化效果
thresholds = [80, int(gray_np.mean()), 170]

plt.figure(figsize=(12, 4))
plt.subplot(1, len(thresholds) + 1, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('原始灰度图')
plt.axis('off')

for i, t in enumerate(thresholds, start=2):
    binary_np = (gray_np >= t).astype(np.uint8) * 255
    plt.subplot(1, len(thresholds) + 1, i)
    plt.imshow(binary_np, cmap='gray')
    plt.title(f'阈值={t}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print('提示：阈值越高，白色区域通常会越少。')


## Step 4 数据增强

数据增强就是“对同一张图片做不同变化”，让 AI 见到更多样本。

常见方法：
- 翻转
- 旋转
- 改亮度
- 裁剪再缩放

作用：提升模型面对新图片时的适应能力。


In [ ]:
# 第5步：对同一张图做四种增强
aug_images = {
    '原图': img_rgb,
    '水平翻转': ImageOps.mirror(img_rgb),
    '旋转15度': img_rgb.rotate(15, expand=False),
    '变亮1.4倍': ImageEnhance.Brightness(img_rgb).enhance(1.4),
}

# 裁剪中间区域后再缩放回原尺寸，也是常见增强方式
w, h = img_rgb.size
crop_box = (w * 0.1, h * 0.1, w * 0.9, h * 0.9)
cropped = img_rgb.crop(tuple(map(int, crop_box))).resize((w, h))
aug_images['中心裁剪+缩放'] = cropped

plt.figure(figsize=(12, 6))
for idx, (name, aug_img) in enumerate(aug_images.items(), start=1):
    plt.subplot(2, 3, idx)
    plt.imshow(aug_img)
    plt.title(name)
    plt.axis('off')

plt.tight_layout()
plt.show()


## 课堂练习

1. 把二值化阈值改成 `50`、`120`、`200`，比较结果。
2. 新增一种增强方式：例如旋转 `-20` 度或降低对比度。
3. 思考：如果拍照光线很暗，哪种预处理最重要？为什么？

常见坑：
- 忘记把像素从 `0~255` 转成 `0~1`。
- 灰度图显示时没加 `cmap='gray'`，会看到奇怪颜色。


In [ ]:
# 练习答案脚手架：你可以在这里实现一个“完整预处理函数”
def preprocess_pipeline(image: Image.Image, threshold: int = 120):
    # 输入: PIL 图片
    # 输出: 标准化数组、灰度图、二值图
    # 1) 标准化
    arr = np.array(image).astype(np.float32) / 255.0
    m = arr.mean(axis=(0, 1), keepdims=True)
    s = arr.std(axis=(0, 1), keepdims=True) + 1e-6
    arr_norm = (arr - m) / s

    # 2) 灰度
    gray = ImageOps.grayscale(image)
    gray_arr = np.array(gray)

    # 3) 二值化
    binary_arr = (gray_arr >= threshold).astype(np.uint8) * 255
    return arr_norm, gray, binary_arr

arr_norm, gray_img, bin_img = preprocess_pipeline(img_rgb, threshold=120)
print('预处理完成：', arr_norm.shape, gray_img.size, bin_img.shape)
